# LeetCode 2185 — Counting Words With a Given Prefix bằng Python

Link đề: https://leetcode.com/problems/counting-words-with-a-given-prefix/ · Độ khó: **Easy** · Chủ đề: Array, String, String Matching

Skeleton LeetCode:

```python
class Solution:
    def prefixCount(self, words: List[str], pref: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **tiền tố (prefix)** của một chuỗi là gì và phân biệt với hậu tố, chuỗi con.
2. Biết 3 cách kiểm tra "`w` bắt đầu bằng `pref`": **cắt lát** `w[:k] == pref`, **`str.startswith`**, và **so từng ký tự** có dừng sớm.
3. Viết vòng lặp **đếm có điều kiện** gọn bằng `sum(... for ...)`.
4. Làm quen **Trie** và **sắp xếp + bisect** (cây tiền tố) với bộ đếm `pass_count` để trả lời truy vấn prefix rất nhanh khi có nhiều truy vấn.
5. So sánh 4 cách, trace từng bước, test ngẫu nhiên đối chiếu và tránh các lỗi hay gặp (nhầm `in` với `startswith`, quên từ ngắn hơn `pref`).

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` và một chuỗi `pref`. Hãy trả về **số chuỗi trong `words` có `pref` là tiền tố**.

Tiền tố của chuỗi `s` là một chuỗi con **liên tiếp xuất hiện ở đầu** `s`. Ví dụ `"at"` là tiền tố của `"attention"` nhưng không phải của `"practice"`.

| Ví dụ | words | pref | Output |
|---|---|---|---|
| 1 | `["pay","attention","practice","attend"]` | `"at"` | `2` |
| 2 | `["leetcode","win","loops","success"]` | `"code"` | `0` |

- Ví dụ 1: chỉ `"attention"` và `"attend"` bắt đầu bằng `"at"`.
- Ví dụ 2: `"leetcode"` **chứa** `"code"` nhưng ở cuối chứ không ở đầu nên không tính.

Constraints: `1 <= len(words) <= 100`, `1 <= len(words[i]), len(pref) <= 100`, chỉ gồm chữ thường tiếng Anh.

In [1]:
examples = [
    (["pay", "attention", "practice", "attend"], "at", 2),
    (["leetcode", "win", "loops", "success"], "code", 0),
]
for words, pref, expected in examples:
    got = [w for w in words if w.startswith(pref)]
    print("pref =", pref, "| mong đợi =", expected, "| các từ khớp:", got)

pref = at | mong đợi = 2 | các từ khớp: ['attention', 'attend']
pref = code | mong đợi = 0 | các từ khớp: []


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự, có chỉ số và lát cắt

`s[i]` là ký tự thứ `i` (từ 0). `s[a:b]` là lát cắt từ `a` đến **trước** `b`. `s[:k]` là `k` ký tự đầu. Nếu `k` lớn hơn độ dài, Python **không báo lỗi**, chỉ trả về cả chuỗi.

In [2]:
s = "attention"
print(s[0], s[1], s[-1])
print(s[:2], s[:5], s[:100])
print(len(s), len(s[:2]))

a t n
at atten attention
9 2


### 3.2. So sánh hai chuỗi bằng `==`

`"at" == "at"` là `True`; chuỗi phân biệt hoa thường và độ dài. Vì `s[:k]` luôn có tối đa `k` ký tự, nên nếu `s` ngắn hơn `k` thì `s[:k]` ngắn hơn `pref` và phép `==` tự cho `False` — đó chính là cách xử lý từ ngắn hơn tiền tố.

In [3]:
print("at" == "at", "at" == "At", "at" == "att")
print("a"[:2] == "at")      # từ chỉ có 1 ký tự, không thể có tiền tố 'at'

True False False
False


### 3.3. `str.startswith`

`w.startswith(p)` trả về `True` nếu `w` bắt đầu bằng `p`. Nó viết sẵn bằng C nên nhanh và rõ nghĩa. Phân biệt với toán tử `in`: `p in w` chỉ hỏi "`p` có xuất hiện **ở đâu đó** trong `w`".

In [4]:
print("attention".startswith("at"), "leetcode".startswith("code"), "code" in "leetcode")
print("attention".endswith("ion"), "ab".startswith("abc"))

True False True
True False


### 3.4. Đếm có điều kiện

Trong Python `True == 1`, nên `sum(điều_kiện for x in ds)` đếm số phần tử thỏa điều kiện. Cách này tương đương một vòng `for` có biến `count += 1`.

In [5]:
nums = [3, 8, 1, 10, 7]
print(sum(1 for x in nums if x > 5))
print(sum(x > 5 for x in nums))     # True được cộng như 1
count = 0
for x in nums:
    if x > 5:
        count += 1
print(count)

3
3
3


### 3.5. `dict` lồng nhau (chuẩn bị cho Trie)

Một **Trie** là cây mà mỗi nút có một `dict` từ ký tự sang nút con. Ta cần biết tạo, kiểm tra và đi xuống một dict lồng nhau.

In [6]:
node = {}
node["a"] = {}
node["a"]["t"] = {}
print(node, "a" in node, "b" in node)
cur = node
for ch in "at":
    cur = cur[ch]
print(cur)

{'a': {'t': {}}} True False
{}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 với `pref = "at"` (độ dài `k = 2`): lấy **2 ký tự đầu** của từng từ rồi so với `"at"`.

| từ | 2 ký tự đầu | bằng `"at"`? |
|---|---|---|
| `pay` | `pa` | không |
| `attention` | `at` | **có** |
| `practice` | `pr` | không |
| `attend` | `at` | **có** |

Đếm được 2. Mỗi từ chỉ cần nhìn **đầu từ**, không cần quét cả từ. Đây là bài "lọc rồi đếm" đơn giản nhất: một vòng qua `words` và một phép so sánh đầu chuỗi.

In [7]:
words, pref = ["pay", "attention", "practice", "attend"], "at"
k = len(pref)
for w in words:
    print(f"{w:10s} -> {w[:k]!r:6} == {pref!r}? {w[:k] == pref}")

pay        -> 'pa'   == 'at'? False
attention  -> 'at'   == 'at'? True
practice   -> 'pr'   == 'at'? False
attend     -> 'at'   == 'at'? True


## 5. Cách 1 — Brute Force / baseline: so từng ký tự bằng vòng lặp

Ý tưởng "thủ công" nhất: với mỗi từ, so từng vị trí `i` của `pref` với `w[i]`. Nếu từ ngắn hơn `pref` hoặc có ký tự lệch thì **loại**. Chưa cần mẹo nào của Python.

In [8]:
def has_prefix_manual(w, pref):
    if len(w) < len(pref):
        return False
    for i in range(len(pref)):
        if w[i] != pref[i]:
            return False
    return True

print(has_prefix_manual("attention", "at"), has_prefix_manual("a", "at"), has_prefix_manual("leetcode", "code"))

True False False


In [9]:
from typing import List

class SolutionBrute:
    def prefixCount(self, words: List[str], pref: str) -> int:
        count = 0
        for w in words:
            ok = len(w) >= len(pref)
            for i in range(len(pref) if ok else 0):
                if w[i] != pref[i]:
                    ok = False
                    break
            if ok:
                count += 1
        return count

print(SolutionBrute().prefixCount(["pay", "attention", "practice", "attend"], "at"))
print(SolutionBrute().prefixCount(["leetcode", "win", "loops", "success"], "code"))

2
0


**Độ phức tạp:** gọi `n = len(words)`, `m = len(pref)`. Mỗi từ so tối đa `m` ký tự nên **Time `O(n·m)`**, **Space `O(1)`**.

**Nút thắt:** code đúng nhưng dài và dễ sai chỉ số (quên kiểm tra độ dài sẽ gây `IndexError`). Python có công cụ dựng sẵn làm đúng việc này, ta dùng nó ở các cách sau.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: **từ `w` có tiền tố `pref` khi và chỉ khi `m` ký tự đầu của `w` chính là `pref`**. Quy tắc:

- nếu `w[:m] == pref` thì cộng 1;
- nếu `w` ngắn hơn `m` thì `w[:m]` ngắn hơn `pref` nên so sánh tự `False`, không cần xử lý riêng;
- tổng thời gian không thể tốt hơn đọc đầu mỗi từ, tức `O(n·m)`.

Nếu có **rất nhiều truy vấn `pref`** trên cùng `words`, ta nên xây Trie một lần rồi trả lời mỗi truy vấn trong `O(m)` (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Cắt lát `w[:m] == pref`

Với mỗi từ lấy `m` ký tự đầu rồi so sánh. Việc so sánh chuỗi do Python lo, ta chỉ đếm.

In [10]:
from typing import List

class SolutionSlice:
    def prefixCount(self, words: List[str], pref: str) -> int:
        m = len(pref)
        count = 0
        for w in words:
            if w[:m] == pref:
                count += 1
        return count

print(SolutionSlice().prefixCount(["pay", "attention", "practice", "attend"], "at"))

2


**Độ phức tạp:** Time `O(n·m)` (mỗi lát cắt tạo chuỗi mới dài tối đa `m`), Space `O(m)` tạm thời.

### Cách 3 — `startswith` + `sum`

`startswith` không tạo chuỗi mới và dừng ngay khi gặp ký tự lệch. Gói vào `sum` cho một dòng.

In [11]:
from typing import List

class SolutionStartsWith:
    def prefixCount(self, words: List[str], pref: str) -> int:
        return sum(w.startswith(pref) for w in words)

print(SolutionStartsWith().prefixCount(["leetcode", "win", "loops", "success"], "code"))
print(SolutionStartsWith().prefixCount(["pay", "attention", "practice", "attend"], "at"))

0
2


**Độ phức tạp:** Time `O(n·m)` trường hợp xấu nhất, nhưng thực tế thường dừng sớm; Space `O(1)`. Đây là cách ngắn gọn và được khuyến nghị khi nộp bài.

### Cách 4 — Trie với bộ đếm `pass_count`

Xây cây tiền tố: mỗi nút có `children` (dict) và `pass_count` = **số từ đi qua nút đó**. Chèn mọi từ vào Trie; khi muốn đếm từ có tiền tố `pref`, ta đi theo `pref` trong cây và đọc `pass_count` ở nút cuối. Nếu giữa đường thiếu ký tự thì đáp án `0`.

In [12]:
from typing import List

class TrieNode:
    def __init__(self):
        self.children = {}
        self.pass_count = 0

class SolutionTrie:
    def prefixCount(self, words: List[str], pref: str) -> int:
        root = TrieNode()
        for w in words:
            node = root
            for ch in w:
                if ch not in node.children:
                    node.children[ch] = TrieNode()
                node = node.children[ch]
                node.pass_count += 1
        node = root
        for ch in pref:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.pass_count

print(SolutionTrie().prefixCount(["pay", "attention", "practice", "attend"], "at"))
print(SolutionTrie().prefixCount(["leetcode", "win", "loops", "success"], "code"))

2
0


**Độ phức tạp:** Xây cây `O(tổng độ dài các từ)`, mỗi truy vấn `O(m)`; Space `O(tổng độ dài)`. Với **một** truy vấn như đề này thì Trie **không nhanh hơn** `startswith`, nhưng rất lợi khi có nhiều truy vấn (autocomplete).

### Cách 5 (mở rộng) — Sắp xếp rồi chặt nhị phân với `bisect`

Khi có **nhiều truy vấn**, ta có thể sắp xếp `words` một lần. Trong danh sách đã sắp xếp theo thứ tự từ điển, **mọi từ có cùng tiền tố nằm liền nhau thành một đoạn**. Ta tìm đầu đoạn bằng `bisect_left(words, pref)` và cuối đoạn bằng `bisect_left(words, pref + "{")` (ký tự `"{"` đứng ngay sau `"z"` trong bảng ASCII nên `pref + "{"` lớn hơn mọi từ bắt đầu bằng `pref`). Hiệu hai vị trí là đáp án.

Ví dụ: `sorted = ["attend","attention","pay","practice"]`, `pref = "at"`: `bisect_left(.., "at") = 0`, `bisect_left(.., "at{") = 2` → đáp án `2 - 0 = 2`.

In [13]:
from bisect import bisect_left
from typing import List

class SolutionBisect:
    def prefixCount(self, words: List[str], pref: str) -> int:
        arr = sorted(words)
        lo = bisect_left(arr, pref)
        hi = bisect_left(arr, pref + "{")
        return hi - lo

print(SolutionBisect().prefixCount(["pay", "attention", "practice", "attend"], "at"))
print(SolutionBisect().prefixCount(["leetcode", "win", "loops", "success"], "code"))
print(sorted(["pay", "attention", "practice", "attend"]))

2
0
['attend', 'attention', 'pay', 'practice']


**Độ phức tạp:** sắp xếp `O(n·m' log n)` (với `m'` là độ dài từ), mỗi truy vấn `O(m·log n)`; Space `O(n)`. Chỉ đáng dùng khi truy vấn nhiều; với một truy vấn thì `startswith` đơn giản hơn nhiều. Lưu ý ràng buộc **chỉ chữ thường** là điều kiện để mẹo `"{"` đúng.

## 8. TRACE từng bước

Trace Cách 2 trên `words = ["pay","attention","practice","attend"]`, `pref = "at"`:

| bước | từ `w` | `w[:2]` | khớp? | `count` |
|---|---|---|---|---|
| 1 | pay | pa | không | 0 |
| 2 | attention | at | có | 1 |
| 3 | practice | pr | không | 1 |
| 4 | attend | at | có | 2 |

In [14]:
def trace_slice(words, pref):
    m = len(pref)
    count = 0
    for step, w in enumerate(words, 1):
        head = w[:m]
        ok = head == pref
        if ok:
            count += 1
        print(f"bước {step}: w={w!r:12} w[:{m}]={head!r:6} khớp={ok!s:5} count={count}")
    return count

print("kết quả:", trace_slice(["pay", "attention", "practice", "attend"], "at"))

bước 1: w='pay'        w[:2]='pa'   khớp=False count=0
bước 2: w='attention'  w[:2]='at'   khớp=True  count=1
bước 3: w='practice'   w[:2]='pr'   khớp=False count=1
bước 4: w='attend'     w[:2]='at'   khớp=True  count=2
kết quả: 2


In [15]:
def trace_trie(words, pref):
    root = {}
    counts = {}
    for w in words:
        path = ""
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
            path += ch
            counts[path] = counts.get(path, 0) + 1
    print("pass_count của một số tiền tố:", {p: counts[p] for p in ["a", "at", "att", "p", "pa", "pr"] if p in counts})
    print("đáp án cho", repr(pref), "=", counts.get(pref, 0))

trace_trie(["pay", "attention", "practice", "attend"], "at")

pass_count của một số tiền tố: {'a': 2, 'at': 2, 'att': 2, 'p': 2, 'pa': 1, 'pr': 1}
đáp án cho 'at' = 2


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def prefixCount(self, words: List[str], pref: str) -> int:
        return sum(w.startswith(pref) for w in words)

print(Solution().prefixCount(["pay", "attention", "practice", "attend"], "at"))
print(Solution().prefixCount(["leetcode", "win", "loops", "success"], "code"))

2
0


## 10. Vì sao thuật toán đúng

- **Định nghĩa:** `w` có tiền tố `pref` ⇔ `len(w) >= len(pref)` và `w[i] == pref[i]` với mọi `i < len(pref)` ⇔ `w[:len(pref)] == pref`. Cả 5 cách đều kiểm tra đúng điều kiện này (cách 5 kiểm tra gián tiếp qua thứ tự từ điển: mọi từ nằm trong khoảng `[pref, pref + "{")` đều bắt đầu bằng `pref`, và ngược lại).
- **Đếm đúng:** mỗi phần tử của `words` được xét đúng một lần và cộng tối đa 1, nên kết quả là số từ thỏa. Từ trùng nhau vẫn được đếm riêng vì ta đếm theo vị trí, không theo giá trị.
- **Trie:** `pass_count` của nút ứng với chuỗi `p` bằng số từ có `p` làm tiền tố, vì mỗi từ tăng `pass_count` của **mọi nút trên đường đi** từ gốc xuống.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(words)`, `m = len(pref)`, `L` = tổng độ dài các từ.

| Cách | Time | Space |
|---|---|---|
| 1. So từng ký tự | `O(n·m)` | `O(1)` |
| 2. Cắt lát | `O(n·m)` | `O(m)` |
| 3. startswith | `O(n·m)` | `O(1)` |
| 4. Trie | `O(L + m)` | `O(L)` |
| 5. Sort + bisect | `O(n log n)` + `O(m log n)` mỗi truy vấn | `O(n)` |

Với giới hạn đề (`n, m ≤ 100`) mọi cách đều gần như tức thời.

## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ và test ngẫu nhiên (có `random.seed`) để đối chiếu **cả 6 cài đặt**.

In [17]:
import random

solvers = [SolutionBrute(), SolutionSlice(), SolutionStartsWith(), SolutionTrie(), SolutionBisect(), Solution()]

fixed = [
    (["pay", "attention", "practice", "attend"], "at", 2),
    (["leetcode", "win", "loops", "success"], "code", 0),
    (["a"], "a", 1),                       # từ đúng bằng pref
    (["a", "ab"], "abc", 0),               # tất cả đều ngắn hơn pref
    (["abc", "abc", "abd"], "ab", 3),      # trùng lặp vẫn đếm
    (["xabc"], "abc", 0),                  # phản ví dụ: chứa nhưng không là tiền tố
    (["zzz", "zz", "z"], "zz", 2),
]
for words, pref, expected in fixed:
    for s in solvers:
        assert s.prefixCount(words, pref) == expected, (type(s).__name__, words, pref)
print("test cố định: OK")

random.seed(2185)
for _ in range(500):
    n = random.randint(1, 12)
    words = ["".join(random.choice("ab") for _ in range(random.randint(1, 6))) for _ in range(n)]
    pref = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    expected = sum(1 for w in words if w[:len(pref)] == pref and len(w) >= len(pref))
    results = {type(s).__name__: s.prefixCount(words, pref) for s in solvers}
    assert all(v == expected for v in results.values()), (words, pref, results)
print("500 test ngẫu nhiên: OK")

test cố định: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `in` thay cho `startswith`

`in` chỉ kiểm tra có xuất hiện ở đâu đó, không phải ở đầu.

In [18]:
def wrong1(words, pref):
    return sum(pref in w for w in words)

def right1(words, pref):
    return sum(w.startswith(pref) for w in words)

words, pref = ["leetcode", "win", "loops", "success"], "code"
print("sai:", wrong1(words, pref), "| đúng:", right1(words, pref))

sai: 1 | đúng: 0


### Lỗi 2: Truy cập `w[i]` khi từ ngắn hơn `pref`

Nếu không kiểm tra độ dài, `w[i]` gây `IndexError`.

In [19]:
def wrong2(w, pref):
    for i in range(len(pref)):
        if w[i] != pref[i]:
            return False
    return True

def right2(w, pref):
    if len(w) < len(pref):
        return False
    return all(w[i] == pref[i] for i in range(len(pref)))

try:
    wrong2("a", "at")
except IndexError as e:
    print("sai -> IndexError:", e)
print("đúng:", right2("a", "at"), right2("attend", "at"))

sai -> IndexError: string index out of range
đúng: False True


### Lỗi 3: Dùng `set(words)` làm mất từ trùng nhau

Đề đếm số **phần tử** trong mảng, nên từ trùng được đếm nhiều lần.

In [20]:
def wrong3(words, pref):
    return sum(w.startswith(pref) for w in set(words))

words = ["ab", "ab", "ac"]
print("sai:", wrong3(words, "a"), "| đúng:", sum(w.startswith("a") for w in words))

sai: 2 | đúng: 3


### Lỗi 4: Nhầm tiền tố với hậu tố hoặc chuỗi con

`endswith` kiểm tra hậu tố; `find(p) == 0` mới tương đương `startswith`. Hãy đọc kỹ đề: "ở đầu chuỗi".

In [21]:
w = "attention"
print(w.startswith("at"), w.endswith("at"), w.find("at") == 0, w.find("ten") == 0)

True False True False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "bắt đầu bằng / tiền tố" → nghĩ ngay `startswith` hoặc cắt lát `s[:k]`.
- "Đếm số phần tử thỏa điều kiện" → `sum(cond for x in a)`.
- Có **nhiều truy vấn tiền tố** hoặc cần gợi ý tự hoàn thành → Trie hoặc sắp xếp rồi chặt nhị phân.
- Muốn nhanh với dữ liệu nhỏ: đừng cầu kỳ, `startswith` là đủ; Trie chỉ để rèn luyện cấu trúc.
- Nhớ: `startswith` còn nhận **tuple** các tiền tố, ví dụ `w.startswith(("at", "pr"))`.

In [22]:
print("practice".startswith(("at", "pr")), "pay".startswith(("at", "pr")))

True False


## 15. Biến thể của bài

1. Đếm từ có **hậu tố** `suff` (dùng `endswith`) — LeetCode 3042 là bản đếm cặp tiền tố-hậu tố.
2. Trả về **danh sách** các từ có tiền tố `pref` thay vì số lượng.
3. Cho `q` truy vấn tiền tố: dùng Trie hoặc sort + `bisect` để mỗi truy vấn nhanh.
4. LeetCode 2255 — đếm số từ trong `words` là **tiền tố của `s`** (đảo vai trò).
5. Bỏ phân biệt hoa thường: đưa về `lower()` trước khi so.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng ký tự | vòng lặp + `break` | `O(n·m)` | `O(1)` | học nền tảng, ngôn ngữ không có `startswith` |
| 2. Cắt lát | `w[:m] == pref` | `O(n·m)` | `O(m)` | dễ hiểu, ngắn |
| 3. `startswith` | hàm dựng sẵn + `sum` | `O(n·m)` | `O(1)` | **khuyến nghị khi nộp** |
| 4. Trie | đếm lượt đi qua nút | `O(L + m)` | `O(L)` | nhiều truy vấn trên cùng tập từ |
| 5. Sort + bisect | tiền tố thành một đoạn liền nhau | `O(n log n)` + `O(m log n)`/truy vấn | `O(n)` | nhiều truy vấn, không muốn viết Trie |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Dòng sau in ra gì? `print("banana"[:3] == "ban", "ba"[:3] == "ban")`

**Bài 2 (tính tay).** `words = ["apple","app","ap","a","apply"]`, `pref = "app"`. Đáp án là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành hàm đếm từ có **hậu tố** `suff`.

In [23]:
def count_suffix(words, suff):
    # TODO: trả về số từ trong words kết thúc bằng suff
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết hàm `prefix_words(words, pref)` trả về **danh sách** các từ có tiền tố `pref` (giữ nguyên thứ tự).

In [24]:
def prefix_words(words, pref):
    # TODO: tự viết
    pass

**Bài 5 (mini challenge).** Cho `words` và danh sách `queries` các tiền tố. Trả về danh sách số lượng tương ứng cho mỗi truy vấn. Gợi ý: dùng Trie hoặc `Counter` của mọi tiền tố của mọi từ.

In [25]:
def prefix_counts(words, queries):
    # TODO: trả về [số từ có tiền tố q for q in queries]
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[:3]` của `"ba"` chỉ có 2 ký tự. Bài 2: liệt kê từ nào bắt đầu bằng `app`.
- **Mức 2 (vừa):** Bài 3 thay `startswith` bằng `endswith`; Bài 4 dùng list comprehension có `if`.
- **Mức 3 (sâu):** Bài 5 — tạo `Counter` mọi tiền tố: với mỗi từ `w` và mỗi `i` từ 1 đến `len(w)`, tăng `cnt[w[:i]]`. Truy vấn thành `cnt[q]` trong `O(1)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
from collections import Counter

# Bài 1 và 2
assert ("banana"[:3] == "ban", "ba"[:3] == "ban") == (True, False)
assert sum(w.startswith("app") for w in ["apple", "app", "ap", "a", "apply"]) == 3

# Bài 3
def count_suffix(words, suff):
    return sum(w.endswith(suff) for w in words)

# Bài 4
def prefix_words(words, pref):
    return [w for w in words if w.startswith(pref)]

# Bài 5
def prefix_counts(words, queries):
    cnt = Counter()
    for w in words:
        for i in range(1, len(w) + 1):
            cnt[w[:i]] += 1
    return [cnt[q] for q in queries]

assert count_suffix(["running", "jumping", "walk", "ing"], "ing") == 3
assert prefix_words(["pay", "attention", "attend"], "at") == ["attention", "attend"]
words = ["pay", "attention", "practice", "attend"]
queries = ["at", "p", "pr", "x", "attend"]
assert prefix_counts(words, queries) == [SolutionBrute().prefixCount(words, q) for q in queries]
print(prefix_counts(words, queries))
print("tất cả bài tập OK")

[2, 2, 1, 0, 1]
tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
class Solution:
    def prefixCount(self, words: List[str], pref: str) -> int:
        return sum(w.startswith(pref) for w in words)
```

**Checklist**

- [ ] Có dùng `startswith` (chứ không phải `in`) không?
- [ ] Từ ngắn hơn `pref` có được xử lý (trả về `False`, không `IndexError`)?
- [ ] Từ trùng nhau có được đếm đủ (không dùng `set`)?
- [ ] Đã thử `pref` dài hơn mọi từ → kết quả `0`?
- [ ] Đã thử từ đúng bằng `pref` → được tính?
- [ ] Đã thử phản ví dụ `"leetcode"` với `"code"` → `0`?
- [ ] Signature đúng `prefixCount(self, words, pref)` và có `from typing import List`?